In [1]:
import os
import sys
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.impute import SimpleImputer
import joblib
import warnings
warnings.filterwarnings("ignore")

sys.path.append("../src")
from feature_extraction.audio_features import extract_features

In [2]:
train_labels = pd.read_csv("../data/daic_woz/train_split_Depression_AVEC2017.csv")
dev_labels   = pd.read_csv("../data/daic_woz/dev_split_Depression_AVEC2017.csv")
all_labels   = pd.concat([train_labels, dev_labels], ignore_index=True)

# Keep only what we need
all_labels = all_labels[["Participant_ID", "PHQ8_Binary", "PHQ8_Score", "Gender"]]
print(all_labels.head())
print(f"\nTotal participants in labels: {len(all_labels)}")
print(f"Label distribution: {all_labels['PHQ8_Binary'].value_counts().to_dict()}")

   Participant_ID  PHQ8_Binary  PHQ8_Score  Gender
0             303            0           0       0
1             304            0           6       0
2             305            0           7       1
3             310            0           4       1
4             312            0           2       1

Total participants in labels: 142
Label distribution: {0: 100, 1: 42}


In [3]:
AUDIO_DIR   = "../data/daic_woz/audio"
SEGMENT_SEC = 10
TARGET_SR   = 22050  # resample to match our pipeline

dataset = []
errors  = []

audio_files = sorted([f for f in os.listdir(AUDIO_DIR) if f.endswith(".wav")])
print(f"Found {len(audio_files)} audio files\n")

for filename in audio_files:
    participant_id = int(filename.split("_")[0])

    # Get label
    row = all_labels[all_labels["Participant_ID"] == participant_id]
    if row.empty:
        print(f"  No label found for {participant_id} — skipping")
        continue

    label     = int(row["PHQ8_Binary"].values[0])
    phq_score = float(row["PHQ8_Score"].values[0])
    gender    = int(row["Gender"].values[0]) if not pd.isna(row["Gender"].values[0]) else -1

    filepath = os.path.join(AUDIO_DIR, filename)

    # Load and resample
    try:
        y, sr = librosa.load(filepath, sr=TARGET_SR, mono=True)
    except Exception as e:
        errors.append(f"Load error {filename}: {e}")
        continue

    # Segment into 10-second chunks
    segment_len = SEGMENT_SEC * TARGET_SR
    n_segments  = len(y) // segment_len
    valid_segs  = 0

    for i in range(n_segments):
        start = i * segment_len
        end   = start + segment_len
        chunk = y[start:end]

        # Skip silent or near-silent segments
        if np.abs(chunk).mean() < 0.001:
            continue

        # Save chunk as temp wav
        tmp_path = f"/tmp/chunk_{participant_id}_{i}.wav"
        sf.write(tmp_path, chunk, TARGET_SR)

        try:
            features = extract_features(tmp_path)
            features["participant_id"] = participant_id
            features["segment_idx"]    = i
            features["label"]          = label
            features["phq8_score"]     = phq_score
            features["gender"]         = gender
            dataset.append(features)
            valid_segs += 1
        except Exception as e:
            errors.append(f"Feature error {participant_id} seg {i}: {e}")
        finally:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)

    print(f"  {participant_id:>4} | label={label} | PHQ8={phq_score:>4.1f} | segments={valid_segs}")

print(f"\nTotal segments extracted : {len(dataset)}")
print(f"Errors                   : {len(errors)}")
if errors:
    for e in errors[:5]:
        print(f"  {e}")

Found 52 audio files

  No label found for 300 — skipping
  No label found for 301 — skipping
   302 | label=0 | PHQ8= 4.0 | segments=75
   304 | label=0 | PHQ8= 6.0 | segments=79
   305 | label=0 | PHQ8= 7.0 | segments=170
   312 | label=0 | PHQ8= 2.0 | segments=76
   313 | label=0 | PHQ8= 7.0 | segments=64
   315 | label=0 | PHQ8= 2.0 | segments=97
   316 | label=0 | PHQ8= 6.0 | segments=70
   317 | label=0 | PHQ8= 8.0 | segments=67
   318 | label=0 | PHQ8= 3.0 | segments=51
   319 | label=1 | PHQ8=13.0 | segments=40
   320 | label=1 | PHQ8=11.0 | segments=76
   321 | label=1 | PHQ8=20.0 | segments=49
   322 | label=0 | PHQ8= 5.0 | segments=89
   324 | label=0 | PHQ8= 5.0 | segments=67
   325 | label=1 | PHQ8=10.0 | segments=78
   326 | label=0 | PHQ8= 2.0 | segments=57
   327 | label=0 | PHQ8= 4.0 | segments=58
   328 | label=0 | PHQ8= 4.0 | segments=97
   330 | label=1 | PHQ8=12.0 | segments=47
   331 | label=0 | PHQ8= 8.0 | segments=75
   333 | label=0 | PHQ8= 5.0 | segments=80
  

In [4]:
df = pd.DataFrame(dataset)
print(f"Dataset shape: {df.shape}")
print(f"\nLabel distribution (segments):")
print(df["label"].value_counts())
print(f"\nParticipants:")
print(df.groupby(["participant_id","label"])["segment_idx"].count().reset_index().rename(columns={"segment_idx":"segments"}))

Dataset shape: (3791, 24)

Label distribution (segments):
label
1    2209
0    1582
Name: count, dtype: int64

Participants:
    participant_id  label  segments
0              302      0        75
1              304      0        79
2              305      0       170
3              312      0        76
4              313      0        64
5              315      0        97
6              316      0        70
7              317      0        67
8              318      0        51
9              319      1        40
10             320      1        76
11             321      1        49
12             322      0        89
13             324      0        67
14             325      1        78
15             326      0        57
16             327      0        58
17             328      0        97
18             330      1        47
19             331      0        75
20             333      0        80
21             335      1        70
22             336      0        82
23         

In [5]:
os.makedirs("../data/voice_features", exist_ok=True)
df.to_csv("../data/voice_features/daic_woz_depression_dataset.csv", index=False)
print("Saved: data/voice_features/daic_woz_depression_dataset.csv")

Saved: data/voice_features/daic_woz_depression_dataset.csv


In [6]:
feature_cols = [
    "pitch", "spectral_centroid", "zcr",
    "jitter", "shimmer", "hnr",
    "mfcc_1","mfcc_2","mfcc_3","mfcc_4","mfcc_5","mfcc_6","mfcc_7",
    "mfcc_8","mfcc_9","mfcc_10","mfcc_11","mfcc_12","mfcc_13"
]

X = df[feature_cols]
y = df["label"]
groups = df["participant_id"]

imputer = SimpleImputer(strategy="mean")
X_imp   = imputer.fit_transform(X)

# First check participant-level label distribution
participant_labels = df.groupby("participant_id")["label"].first()
print("Participant-level distribution:")
print(participant_labels.value_counts())
print(f"\nDepressed participants: {sorted(participant_labels[participant_labels==1].index.tolist())}")
print(f"Healthy participants:   {sorted(participant_labels[participant_labels==0].index.tolist())}")

# Manual 3-fold split ensuring both classes in each fold
depressed_ids = sorted(participant_labels[participant_labels==1].index.tolist())
healthy_ids   = sorted(participant_labels[participant_labels==0].index.tolist())

print(f"\nDepressed count: {len(depressed_ids)}")
print(f"Healthy count:   {len(healthy_ids)}")

# Split each group into 3 folds
import math
def split_into_folds(ids, n_folds=3):
    size = math.ceil(len(ids) / n_folds)
    return [ids[i*size:(i+1)*size] for i in range(n_folds)]

dep_folds = split_into_folds(depressed_ids, 3)
hlt_folds = split_into_folds(healthy_ids, 3)

model_cv = RandomForestClassifier(n_estimators=200, random_state=42)
cv_acc, cv_auc = [], []

for fold in range(3):
    test_ids  = dep_folds[fold] + hlt_folds[fold]
    train_ids = [pid for pid in participant_labels.index if pid not in test_ids]

    test_mask  = df["participant_id"].isin(test_ids)
    train_mask = df["participant_id"].isin(train_ids)

    X_train = X_imp[train_mask]
    X_test  = X_imp[test_mask]
    y_train = y[train_mask]
    y_test  = y[test_mask]

    print(f"\nFold {fold+1}:")
    print(f"  Train: {len(train_ids)} participants, {len(X_train)} segments")
    print(f"  Test:  {len(test_ids)} participants, {len(X_test)} segments")
    print(f"  Test label dist: {dict(y_test.value_counts())}")

    if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
        print(f"  Skipped — only one class present")
        continue

    model_cv.fit(X_train, y_train)
    preds = model_cv.predict(X_test)
    probs = model_cv.predict_proba(X_test)[:,1]

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    cv_acc.append(acc)
    cv_auc.append(auc)
    print(f"  Accuracy : {acc:.4f}")
    print(f"  AUROC    : {auc:.4f}")

print(f"\nSubject-Wise CV Results (DAIC-WoZ Clinical Depression):")
print(f"  Valid folds : {len(cv_acc)} of 3")
print(f"  Accuracy    : {np.mean(cv_acc):.4f} +/- {np.std(cv_acc):.4f}")
print(f"  AUROC       : {np.mean(cv_auc):.4f} +/- {np.std(cv_auc):.4f}")

Participant-level distribution:
label
1    28
0    22
Name: count, dtype: int64

Depressed participants: [319, 320, 321, 325, 330, 335, 338, 339, 344, 345, 346, 347, 348, 350, 351, 352, 353, 355, 356, 362, 367, 376, 377, 380, 381, 386, 388, 389]
Healthy participants:   [302, 304, 305, 312, 313, 315, 316, 317, 318, 322, 324, 326, 327, 328, 331, 333, 336, 340, 341, 343, 357, 358]

Depressed count: 28
Healthy count:   22

Fold 1:
  Train: 32 participants, 2425 segments
  Test:  18 participants, 1366 segments
  Test label dist: {0: 698, 1: 668}
  Accuracy : 0.6193
  AUROC    : 0.6744

Fold 2:
  Train: 32 participants, 2528 segments
  Test:  18 participants, 1263 segments
  Test label dist: {1: 689, 0: 574}
  Accuracy : 0.6097
  AUROC    : 0.7725

Fold 3:
  Train: 36 participants, 2629 segments
  Test:  14 participants, 1162 segments
  Test label dist: {1: 852, 0: 310}
  Accuracy : 0.5215
  AUROC    : 0.5336

Subject-Wise CV Results (DAIC-WoZ Clinical Depression):
  Valid folds : 3 of 3
  A

In [7]:
# Run this in a notebook cell
import pandas as pd
train_labels = pd.read_csv("../data/daic_woz/train_split_Depression_AVEC2017.csv")
dev_labels   = pd.read_csv("../data/daic_woz/dev_split_Depression_AVEC2017.csv")
all_labels   = pd.concat([train_labels, dev_labels])

# Check labels for our extracted participants
extracted_ids = [300, 301, 302, 319, 320, 321, 325, 330, 335, 338, 339,
                 344, 345, 346, 347, 348, 350, 351, 352, 353, 355, 356, 362, 367]
print(all_labels[all_labels["Participant_ID"].isin(extracted_ids)][["Participant_ID","PHQ8_Binary","PHQ8_Score"]])

    Participant_ID  PHQ8_Binary  PHQ8_Score
10             319            1          13
11             320            1          11
12             321            1          20
15             325            1          10
19             330            1          12
22             338            1          15
23             339            1          11
27             344            1          11
28             345            1          15
29             347            1          16
30             348            1          20
31             350            1          11
32             351            1          14
33             352            1          10
34             353            1          11
35             355            1          10
36             356            1          10
40             362            1          20
0              302            0           4
3              335            1          12
4              346            1          23
5              367            1 